# V3：时间轨迹消融

比较 static_lr 与 temporal_lr，保持模型类型、外层折、固定参数和插补规则相同。

先在每个训练折计算变量缺失率，再训练插补、缩放和 LASSO。只为 V3 静态白细胞分类保留既定豁免，时间斜率缺失过多仍被剔除。不会为了凑完整轨迹删除患者。

差值表示变化幅度；斜率表示每小时变化速度；SD表示波动；前后半天差值表示早晚阶段变化；count 表示测量强度。静态均值/min/max与轨迹可能相关，LASSO λ.1se 为可对照的主分析，不保证时间特征一定提高性能。

**balanced 简化计算版**：固定基模型参数、full保留5×10外层验证；LASSO内部5折/30λ，stacking内部3折。与上级严格版使用独立缓存；前6折旧预测不混入。当前没有运行新版本真实患者模型。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 从扩展目录或 notebooks 子目录定位项目，避免硬编码 Windows 盘符。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 dn_mortality_repro_V3_extensions 或其 notebooks 子目录启动。")
sys.path.insert(0, str(ROOT))
print("扩展目录：", ROOT)


In [ ]:
# pilot 是1×3外层小规模试跑；正式实验将 PROFILE 改为 "full"。
# 仅把 RUN_EXPERIMENT 改成 True 才会启动；full可能需要较长时间。
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'temporal_lr']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("尚未启动本Notebook中的模型训练。")


In [ ]:
# 优先查看正式 full 结果；没有结果时打印状态，不重新训练。
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("详细记录：", folder / "report.md")
else:
    print("该模式尚无实验结果。pilot 结果需把 RESULT_PROFILE 改为 pilot 才显示。")


## 实验状态与结论（自动更新）

运行模式：`full`；方法：balanced 固定参数；结果目录：`results/full/27464ba98305edbc`。

LASSO 5折/30λ、stacking 3折；静态与扩展使用同一简化方法。严格版的部分折没有混入。

固定留出集未参与本实验。树模型扩展不需要 stacking 内部交叉拟合；原固定参数及训练折内预处理不变。

Holm 校正覆盖当前已有配对比较；后续增加实验时，校正 p 值可能变化。

```text

  experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
   static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
   static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
  static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690
 temporal_lr             50          0.79564        0.02573          0.60179          0.15263                      0.88260     5687.76804            0.79475            0.58940            0.15262                    0.00314                0.85879
 temporal_rf             50          0.79741        0.02891          0.60594          0.15212                      1.24090       28.89751            0.79605            0.59470            0.15211                    0.00148                1.20217
temporal_xgb             50          0.79284        0.02796          0.60391          0.15300                      0.94159        8.29193            0.79177            0.59437            0.15299                    0.01106                0.91792

```

配对差异：

```text

  experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
   static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
   static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
   static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
  static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
  static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
  static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
 temporal_lr  static_lr  AUROC   -0.00364  0.00867       0.00314 -0.00995 0.00267 -1.15931  49  0.25195 10  5                    1.0
 temporal_lr  static_lr  AUPRC   -0.00560  0.01502       0.00544 -0.01653 0.00532 -1.03037  49  0.30790 10  5                    1.0
 temporal_lr  static_lr  Brier    0.00171  0.00303       0.00110 -0.00049 0.00391  1.56078  49  0.12501 10  5                    1.0
 temporal_rf  static_rf  AUROC   -0.00057  0.00733       0.00265 -0.00591 0.00476 -0.21556  49  0.83022 10  5                    1.0
 temporal_rf  static_rf  AUPRC   -0.00024  0.01373       0.00497 -0.01023 0.00975 -0.04896  49  0.96115 10  5                    1.0
 temporal_rf  static_rf  Brier    0.00034  0.00196       0.00071 -0.00108 0.00177  0.48410  49  0.63047 10  5                    1.0
temporal_xgb static_xgb  AUROC   -0.00220  0.01022       0.00370 -0.00964 0.00523 -0.59579  49  0.55405 10  5                    1.0
temporal_xgb static_xgb  AUPRC   -0.00181  0.02169       0.00785 -0.01759 0.01397 -0.23090  49  0.81836 10  5                    1.0
temporal_xgb static_xgb  Brier    0.00050  0.00389       0.00141 -0.00233 0.00334  0.35547  49  0.72376 10  5                    1.0

```

- `static_rf` 相对 `static_lr`：ΔAUROC=-0.0013，校正95% CI [-0.0119, 0.0093]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `static_xgb` 相对 `static_lr`：ΔAUROC=-0.0042，校正95% CI [-0.0145, 0.0060]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `temporal_lr` 相对 `static_lr`：ΔAUROC=-0.0036，校正95% CI [-0.0099, 0.0027]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `temporal_rf` 相对 `static_rf`：ΔAUROC=-0.0006，校正95% CI [-0.0059, 0.0048]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `temporal_xgb` 相对 `static_xgb`：ΔAUROC=-0.0022，校正95% CI [-0.0096, 0.0052]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

## 2026-10-14 时间消融复核

静态和时间模型各完成5×10个外层折，使用同一开发集、固定模型参数和外层划分，没有读取留出集标签。

| 模型 | AUROC折均值 | AUPRC合并OOF均值 | Brier合并OOF均值 | 校准斜率合并OOF均值 |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| temporal_lr | 0.7956 | 0.5894 | 0.1526 | 0.859 |

`temporal_lr` 相对 `static_lr`：ΔAUROC=-0.0036，Nadeau–Bengio校正95% CI [-0.0099, 0.0027]，p=0.252，Holm校正p=1.000。

**当前定义的0–24h轨迹在本配置下没有显示增量预测价值。** 五次重复合并OOF的AUROC均低于静态LR；50个折中17折更高、33折更低。但校正区间跨0，不能声称性能显著下降。AUPRC较低、Brier较高、校准斜率更偏离1，也没有提供支持保留这组特征的整体证据。

实施核查：每折入选时间特征中位数为4.5个，并非全部被剔除。例如spo2_std在48/50折入选，temperature_count在42/50折入选。测量次数反映检查强度，不能直接解释为生理变化或可干预因素。

本实验只测试当前窗口、汇总轨迹、20%缺失规则、LASSO和固定参数LR；不等于所有时间建模方法都无效。可能的冗余、缺失或增加模型方差是解释假设，尚未得到机制验证。当前不为追求提高AUC而修改窗口或阈值。
